# Text-to-SQL 

The Text-to-SQL Model is an advanced tool designed to generate and execute Clickhouse SQL queries based on natural language questions. It specializes in retrieving information from specified database tables, with a focus on providing accurate and relevant data.


### Using Provider

We will be reusing the `open_ai_provider` we created in [`Using Providers`](Using%20Providers.ipynb).

If you haven't created one yet, create one with the following command:

In [ ]:
CREATE PROVIDER open_ai_provider
ENGINE = OpenAI(
	api_key='sk-proj-xxx',
	model_name = 'gpt-3.5-turbo'
);

## Creating Tools required for Text-to-SQL Model

### Semantics
First we create `get_semantics`, which uses the built-in table function `semantics()`, which returns the schemas of all the tables in the database.

In [ ]:
CREATE VIEW get_semantics() AS 
(SELECT * FROM semantics());

### Query Validator
This model validates the SQL query generated by the `query model` before it gets executed.

In [ ]:
CREATE PROMPT query_validator(
    system "You are clickhouse SQL query validator.
    Take in the SQL query and return True if query is correct. If it is not correct return False and explain what went wrong",
    human "{{query}}"
)

In [ ]:
CREATE MODEL query_validator(
    query)
USING open_ai_provider(model_name = 'gpt-4o')
PROMPT query_validator
SETTINGS retries = 3;;

## Prompt Creation for Query Agent

In [ ]:
CREATE PROMPT query_prompt(
    system "You are an advanced query agent specializing in Clickhouse SQL. Your task is to search the database to retrieve information based on the given question, focusing on the table specified in the input. You must generate Clickhouse-compatible SQL that accurately answers the question.

    Follow these steps for EVERY query:
 
    Step 1: Get Database Schema
    Action: get_semantics tool
    Action Input: all
    Observation: [Output from get_semantics tool showing schema for all tables in the specified schema]

    Step 2: Analyze Schema and Focus on pick tables to query
    Thought: Based on the question, and the available schema for all tables, I will focus on the required tables and identify any potential relationships with other tables if needed.
    Related Tables (if any): [Names of potentially related tables from the schema]
    Reason: [Explanation for why the specified table is appropriate and how any related tables might be useful]
    
    Step 3: Generate and Execute Clickhouse SQL Query
    Thought: I will now generate a Clickhouse SQL query that accurately answers the question, focusing on the specified table. Consider the following:
    - Use the specified table as the primary source of data
    - Use JOINs with other tables only if absolutely necessary to answer the question
    - Never query for all columns from a table, only ask for the few relevant columns given the question
    - Only use the columns available in the tables as shown by the get_semantics tool
    - Limit results to at most 5 unless the user specifies a different number
    - Order results by a relevant column to return the most interesting examples
    - Use only column names visible in the schema description
    - Be careful not to query for columns that do not exist
    - Pay attention to which column is in which table
    - Use appropriate Clickhouse syntax (e.g., `backticks` for identifiers if needed)
    - If the question requires finding specific data (e.g., a city name), use flexible search techniques like ILIKE, lower(), or partial matching

    Step 4: Validate the SQL Query
    Action: query_validator tool
    Action Input: [Your generated ClickHouse SQL query]
    Observation: [true or false]

    Step 5: Execute or Revise Query
    Thought: If the validator returned 'true', proceed to execute the query. If 'false', revise the query and go back to Step 3.
    Action: langdb_raw_query tool
    Action Input: [Your generated Clickhouse SQL query]
    Observation: [Output from langdb_raw_query]

    Always provide your final response in this EXACT format:

    Question: [Original question]
    SQLQuery: [The Clickhouse SQL query you generated]
    SQLResult: [Result of the SQL query]

    If you encounter any errors, include them in your response but maintain the format above.

    Remember: Your query should answer the question accurately, even if it requires complex logic or multiple steps. Focus on the specified table, but consider relationships with other tables if necessary. Use flexible search techniques when looking for specific data. If the user's question is ambiguous or lacks specific details, make reasonable assumptions and state them in your answer.",

    human "{{input}}"
)

## Model Creation
Now, we can create the models that can leverage the tools that were created earlier.

`langdb_raw_query` is the tool which executes the query to retrieve data from the database.

In [ ]:
CREATE MODEL text_to_sql(
    input
) USING open_ai_provider(model_name = 'gpt-4o')
PROMPT query_prompt
TOOLS (
    get_semantics COMMENT 'Tool to get the schemas of the tables in the database',
    langdb_raw_query COMMENT 'Tool to execute SQL query over the database',
    query_validator COMMENT 'Model to Validate the generated SQL query'
)
SETTINGS retries = 3;

### Query Agent Creation

Now you can use it with other models as well as chat with it using the `CHAT` command.

In [ ]:
CHAT text_to_sql